In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
!pip install pandas numpy
!pip install pyECLAT
!pip install mlxtend

In [3]:
import pandas as pd
import numpy as np
from pyECLAT import ECLAT
import time
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules

In [4]:
grocery_data = pd.read_csv('groceries - groceries.csv')
grocery_data.head()



,Item(s),Item 1,Item 2,Item 3,Item 4,Item 5,Item 6,Item 7,Item 8,Item 9,...,Item 23,Item 24,Item 25,Item 26,Item 27,Item 28,Item 29,Item 30,Item 31,Item 32
0,4,citrus fruit,semi-finished bread,margarine,ready soups,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3,tropical fruit,yogurt,coffee,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,whole milk,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,pip fruit,yogurt,cream cheese,meat spreads,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,other vegetables,whole milk,condensed milk,long life bakery product,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
grocery_data.shape


(9835, 33)

created a list for the transactions without NaN/empty items from the data.
Then used a for loop to check each transaction for items that are not a NaN items. 
And then adds them into the list. which becomes list of lists.

the print function prints first 5 transaction items:

transactions[:5]= first five transactions

sep="\n" = to seperate each transaction in new line

'*' = is to apply the seperaction all the transactions 

In [6]:
transactions = []
for i in range(len(grocery_data)):
    transaction = [str(item) for item in grocery_data.iloc[i, 1:] if str(item) != 'nan']
    transactions.append(transaction)
print(*transactions[:5],sep="\n")  

['citrus fruit', 'semi-finished bread', 'margarine', 'ready soups']
['tropical fruit', 'yogurt', 'coffee']
['whole milk']
['pip fruit', 'yogurt', 'cream cheese', 'meat spreads']
['other vegetables', 'whole milk', 'condensed milk', 'long life bakery product']


In [7]:
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df = pd.DataFrame(te_ary, columns=te.columns_)
print(df.shape)

(9835, 169)


used a min sup of 0.01 | 1% 
because needed at least a 100 count of frequent itemsets. and we have 9835 transactions.
thus the min_sup = 100 / 9835 = 0.1016 = 1%

In [8]:
min_sup = 0.01

used time function to record the time of each algotithm to compare them

In [9]:
start_time = time.time()
frequent_itemsets_apriori = apriori(df, min_support=min_sup, use_colnames=True)
apriori_time = time.time() - start_time
rules_apriori = association_rules(frequent_itemsets_apriori, metric="confidence", min_threshold=1.0)


In [10]:
start_time = time.time()
frequent_itemsets_FP = fpgrowth(df, min_support=min_sup, use_colnames=True)
fp_time = time.time() - start_time
rules_FP = association_rules(frequent_itemsets_FP, metric="confidence", min_threshold=10.0)


In [11]:
my_eclat = ECLAT(data=pd.DataFrame(transactions))
start_time = time.time()
rule_indices, rule_supports = my_eclat.fit(min_support=min_sup)
eclat_time = time.time() - start_time
rule_supports

C:\Users\abdoo\AppData\Local\Temp\ipykernel_23412\1194022392.py:1: Pandas4Warning: Constructing a DataFrame from a list of sequences with mismatched lengths is deprecated and will raise in a future version. The shorter sequences are currently padded with NaN; make all sequences the same length before constructing instead.
  my_eclat = ECLAT(data=pd.DataFrame(transactions))


Combination 1 by 1


88it [00:01, 85.00it/s]


Combination 2 by 2


3828it [00:22, 167.20it/s]


Combination 3 by 3


109736it [11:19, 161.53it/s]


{'pip fruit': 0.07564819522114896,
 'hard cheese': 0.024504321301474327,
 'long life bakery product': 0.037417386883579054,
 'mustard': 0.011997966446365024,
 'chewing gum': 0.021047280122013217,
 'frozen meals': 0.02836807320793086,
 'whole milk': 0.25551601423487547,
 'sugar': 0.03385866802236909,
 'detergent': 0.019217081850533807,
 'hamburger meat': 0.033248601931875954,
 'meat': 0.025826131164209457,
 'citrus fruit': 0.08276563294356888,
 'soda': 0.17437722419928825,
 'sausage': 0.09395017793594305,
 'whipped/sour cream': 0.07168276563294357,
 'onions': 0.031011692933401117,
 'salt': 0.010777834265378749,
 'herbs': 0.01626842907981698,
 'semi-finished bread': 0.017691916624300967,
 'specialty bar': 0.027351296390442297,
 'other vegetables': 0.1934926283680732,
 'oil': 0.02806304016268429,
 'domestic eggs': 0.06344687341128623,
 'rolls/buns': 0.18393492628368074,
 'brown bread': 0.06487036095577021,
 'baking powder': 0.017691916624300967,
 'ice cream': 0.025012709710218607,
 'froze

In [14]:
print(f"Apriori Execution Time: {apriori_time:.5f} seconds \nFrequent Itemsets: {len(frequent_itemsets_apriori)}\n")
print(f"FP-Growth Execution Time: {fp_time:.5f} seconds \nFrequent Itemsets: {len(frequent_itemsets_FP)}\n")
print(f"ECLAT Execution Time: {eclat_time:.5f} seconds \nFrequent Itemsets: {len(rule_supports)}")

Apriori Execution Time: 0.10791 seconds 
Frequent Itemsets: 333

FP-Growth Execution Time: 0.09016 seconds 
Frequent Itemsets: 333

ECLAT Execution Time: 626.62315 seconds 
Frequent Itemsets: 332
